# Decoding Deception — Colab training

Thin shell around `src/models/train.py`. Edit the `EXPERIMENT` and `REPO_URL` below.

All training logic lives in the repo so this notebook stays a one-shot.

In [ ]:
# 1. Mount Google Drive (optional, for persistent checkpoints / data)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Sync repo. Defensive: validates Drive path before linking, cleans stale
# symlinks from prior runs, prints what's in Drive if the path is wrong.
import os, subprocess
REPO_URL    = ''                                  # leave '' to use the Drive copy
DRIVE_REPO  = '/content/drive/MyDrive/CCS2'       # adjust to your Drive folder name
WORK        = '/content/CCS2'

if REPO_URL:
    if not os.path.exists(WORK):
        subprocess.check_call(['git', 'clone', REPO_URL, WORK])
else:
    assert os.path.isdir(DRIVE_REPO), (
        f'DRIVE_REPO not found: {DRIVE_REPO}\n'
        f'Top of Drive: {os.listdir("/content/drive/MyDrive/")[:20]}'
    )
    if os.path.lexists(WORK):
        if os.path.islink(WORK):
            os.unlink(WORK)                       # stale symlink from prior run
        else:
            raise SystemExit(f'{WORK} exists and is not a symlink; remove it manually')
    os.symlink(DRIVE_REPO, WORK)

os.chdir(WORK)
print('cwd =', os.getcwd())
print('contents:', os.listdir('.')[:12])

In [ ]:
# 3. Install deps
!pip install -q -r requirements.txt

In [ ]:
# 4. Set caches to Drive so re-runs don't redownload weights
%env HF_HOME=/content/drive/MyDrive/hf_cache
%env TRANSFORMERS_CACHE=/content/drive/MyDrive/hf_cache
%env TORCH_HOME=/content/drive/MyDrive/torch_cache

In [ ]:
# 5. (Optional) Build data pipeline. Skip if data/processed/ is already on Drive.
!python -m src.data.load_existing --all
!python -m src.data.map_labels
!python -m src.data.preprocess
!python -m src.data.split
!python -m src.data.balance

In [ ]:
# 6. Train — 2x2 core experiments + key ablations (sequential).
#
#  2x2 design (H1 + RQ2):
#    Row 1: text-only XLM-R  x  {existing, existing+scraped}
#    Row 2: multimodal fusion x  {existing, existing+scraped}
#
#  Ablations: vision-only sanity, joint jina-clip, modality-dropout,
#             no-balance, no-weak-aux.
#  vision_aux_pretrain skipped: both scraped sources are propaganda-only
#  so the binary aux head sees a single class (see YAML comment).

from pathlib import Path
import subprocess, sys

CORE_EXPERIMENTS = [
    'experiments/text_xlmr_existing.yaml',
    'experiments/text_xlmr_existing_plus_scraped.yaml',
    'experiments/multimodal_latefusion_existing.yaml',
    'experiments/multimodal_latefusion_existing_plus_scraped.yaml',
]

ABLATIONS = [
    'experiments/vision_only.yaml',
    'experiments/joint_jinaclip.yaml',
    'experiments/modality_dropout.yaml',
    'experiments/no_balance.yaml',
    'experiments/no_weak_aux.yaml',
]

ALL_EXPERIMENTS = CORE_EXPERIMENTS + ABLATIONS

failed = []
for exp in ALL_EXPERIMENTS:
    sep = '='*60
    print(f'\n{sep}\nSTARTING: {exp}\n{sep}')
    ret = subprocess.call([sys.executable, '-m', 'src.models.train', '--config', exp])
    if ret != 0:
        print(f'  FAILED (exit {ret}): {exp}')
        failed.append(exp)
    else:
        print(f'  DONE: {exp}')

print('\n=== Summary ===')
done = [e for e in ALL_EXPERIMENTS if e not in failed]
for e in done:
    print(f'  OK   {e}')
for e in failed:
    print(f'  FAIL {e}')


In [ ]:
# 7. Evaluate all completed runs (metrics + bootstrap CIs).
from pathlib import Path
import subprocess, sys

runs_dir = Path('results/runs')
runs = sorted(runs_dir.glob('*'))
if not runs:
    print('No runs found in results/runs/. Did training finish?')
else:
    for run in runs:
        if not (run / 'predictions.parquet').exists():
            print(f'  SKIP (no predictions): {run.name}')
            continue
        print(f'\n--- {run.name} ---')
        subprocess.call([sys.executable, '-m', 'src.evaluation.metrics',
                         '--run-dir', str(run)])
        subprocess.call([sys.executable, '-m', 'src.evaluation.significance',
                         'bootstrap', '--run', str(run), '--B', '1000',
                         '--out', str(run / 'bootstrap_ci.json')])


## Significance, interpretability, error analysis

In [ ]:
# 8a. McNemar H1: multimodal vs text-only (same test items, paired).
import json
from pathlib import Path
import subprocess, sys

runs_dir = Path('results/runs')

def find_run(tag_substr):
    candidates = [r for r in sorted(runs_dir.glob('*')) if tag_substr in r.name
                  and (r / 'predictions.parquet').exists()]
    return candidates[-1] if candidates else None

# H1: multimodal vs text-only, for each data condition
h1_pairs = [
    ('text_xlmr_existing',              'multimodal_latefusion_existing',              'H1_existing'),
    ('text_xlmr_existing_plus_scraped', 'multimodal_latefusion_existing_plus_scraped', 'H1_plus_scraped'),
]
for text_tag, mm_tag, key in h1_pairs:
    rA, rB = find_run(text_tag), find_run(mm_tag)
    if rA is None or rB is None:
        print(f'SKIP {key}: missing run'); continue
    out = f'results/tables/mcnemar_{key}.json'
    ret = subprocess.call([sys.executable, '-m', 'src.evaluation.significance',
                           'mcnemar', '--a', str(rA), '--b', str(rB), '--out', out])
    if ret == 0:
        r = json.load(open(out))
        print(f'{key}: p={r["pvalue"]:.4f}  n={r["n_paired"]}')


In [ ]:
# 8b. McNemar RQ2: existing-only vs +scraped for each modality.
import json
from pathlib import Path
import subprocess, sys

runs_dir = Path('results/runs')

def find_run(tag_substr):
    candidates = [r for r in sorted(runs_dir.glob('*')) if tag_substr in r.name
                  and (r / 'predictions.parquet').exists()]
    return candidates[-1] if candidates else None

rq2_pairs = [
    ('text_xlmr_existing',              'text_xlmr_existing_plus_scraped',              'RQ2_text'),
    ('multimodal_latefusion_existing',   'multimodal_latefusion_existing_plus_scraped',  'RQ2_multimodal'),
]
for base_tag, aug_tag, key in rq2_pairs:
    rA, rB = find_run(base_tag), find_run(aug_tag)
    if rA is None or rB is None:
        print(f'SKIP {key}: missing run'); continue
    out = f'results/tables/mcnemar_{key}.json'
    ret = subprocess.call([sys.executable, '-m', 'src.evaluation.significance',
                           'mcnemar', '--a', str(rA), '--b', str(rB), '--out', out])
    if ret == 0:
        r = json.load(open(out))
        print(f'{key}: p={r["pvalue"]:.4f}  n={r["n_paired"]}')


In [ ]:
# 8c. SHAP interpretability on the feature table (H2 / RQ3).
import subprocess, sys
subprocess.call([sys.executable, '-m', 'src.evaluation.interpretability',
                 '--feature-table', 'data/processed/feature_table.parquet',
                 '--splits', 'data/processed/splits.json',
                 '--out-dir', 'results/figures'])
print('SHAP plots -> results/figures/')


In [ ]:
# 8d. Error analysis: open<->hidden confusions from best multimodal run.
import subprocess, sys
from pathlib import Path

runs_dir = Path("results/runs")
candidates = sorted([
    r for r in runs_dir.glob("*")
    if "multimodal_latefusion_existing_plus_scraped" in r.name
    and (r / "predictions.parquet").exists()
])
if candidates:
    best = candidates[-1]
    subprocess.call([sys.executable, "-m", "src.evaluation.error_analysis",
                     "--run", str(best)])
    print(f"Error analysis -> results/tables/errors_sample.csv  ({best.name})")
else:
    print("No multimodal+scraped run found; run cell 6 first.")


In [ ]:
# 8e. Print 2x2 macro-F1 summary table for the paper.
import json
from pathlib import Path

runs_dir = Path('results/runs')

def best_metric(tag_substr, metric='macro_f1'):
    candidates = [r for r in sorted(runs_dir.glob('*')) if tag_substr in r.name
                  and (r / 'metrics.json').exists()]
    if not candidates:
        return None, None
    best = candidates[-1]
    m = json.load(open(best / 'metrics.json'))
    val = m.get('test', {}).get(metric)
    ci_path = best / 'bootstrap_ci.json'
    ci = json.load(open(ci_path)) if ci_path.exists() else {}
    return val, ci

rows = [
    ('text-only XLM-R',   'text_xlmr_existing',              'text_xlmr_existing_plus_scraped'),
    ('multimodal fusion', 'multimodal_latefusion_existing',   'multimodal_latefusion_existing_plus_scraped'),
]
print(f'{"Model":<22} {"Existing":<24} {"+ Scraped":<24}')
print('-'*70)
for name, ex_tag, aug_tag in rows:
    v1, ci1 = best_metric(ex_tag)
    v2, ci2 = best_metric(aug_tag)
    s1 = f'{v1:.3f} [{ci1.get("ci95_lo",0):.3f}-{ci1.get("ci95_hi",0):.3f}]' if v1 else 'n/a'
    s2 = f'{v2:.3f} [{ci2.get("ci95_lo",0):.3f}-{ci2.get("ci95_hi",0):.3f}]' if v2 else 'n/a'
    print(f'{name:<22} {s1:<24} {s2:<24}')


In [ ]:
# 9. Persist all results back to Drive.
import shutil, time
stamp = int(time.time())
dst = f'/content/drive/MyDrive/propaganda-lt-results-{stamp}.zip'
shutil.make_archive(dst.replace('.zip', ''), 'zip', 'results')
print('wrote', dst)
